# iLASH IBD kinship against GRM, PC-Relate and Weir-Goudet
Scatters are notebook only; the binned table shows bins with 21+ pairs.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
import glob
OUT = single_target(cfg, "ilash")
src = OUT if os.path.exists(OUT) else sorted(glob.glob(OUT.replace("pairs.tsv", "pairs.chr*.tsv")))[-1]   # full run, else the test run
pairs = pd.read_csv(src, sep="\t", dtype={"id1": str, "id2": str})
ids = pd.read_csv(target_items(cfg, "grm_shards")[0][0].split(".grm.bin.")[0] + ".grm.id", sep="\t", header=None, dtype=str)[1]
n = len(ids); pos = pd.Series(np.arange(n), index=ids)
pairs = pairs[pairs["id1"].isin(pos.index) & pairs["id2"].isin(pos.index)]
est = {"GRM / 2": np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")]) / 2}
for lab, st in [("PC-Relate", "pcrelate"), ("Weir-Goudet", "weir_goudet")]:
    u = single_target(cfg, st)
    if os.path.exists(u):
        assert list(pd.read_csv(u.replace(".bin", ".id"), header=None, dtype=str)[0]) == list(ids), st
        est[lab] = np.fromfile(u, dtype="float32")
off = np.ones(n * (n + 1) // 2, bool); off[np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1] = False
a, b = pos[pairs["id1"]].to_numpy(), pos[pairs["id2"]].to_numpy()
t = np.maximum(a, b) * (np.maximum(a, b) + 1) // 2 + np.minimum(a, b)
ibd = np.zeros(off.sum(), "float32")
ibd[np.cumsum(off)[t] - 1] = pairs["kinship_ibd"].to_numpy()   # pairs without segments: 0
est = {k: v[off] for k, v in est.items()}; del off
print(os.path.basename(src), f"{len(pairs):,} pairs with segments of {len(ibd):,}")

In [ ]:
fig, axes = plt.subplots(2, len(est), figsize=(5.5 * len(est), 10), squeeze=False)
seg = ibd > 0
for col, (lab, k) in enumerate(est.items()):   # notebook only
    for row, lim in enumerate([(-0.02, 0.06), (-0.05, 0.6)]):
        ax = axes[row, col]
        ax.scatter(ibd[seg], k[seg], s=1, alpha=0.3, lw=0, rasterized=True)
        ax.plot(lim, lim, "r:", lw=0.8)
        r = np.corrcoef(ibd[seg], k[seg])[0, 1]
        ax.set(xlim=lim, ylim=lim, xlabel="iLASH IBD kinship", ylabel=lab, title=f"{RUN_ID}: pairs with segments, r = {r:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
E = [-1, 1e-9, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 1]
g = pd.cut(pd.Series(ibd), E, labels=["0 (no segments)"] + [f"{lo}-{hi}" for lo, hi in zip(E[1:-1], E[2:])])
tab = pd.DataFrame({"ibd_bin": g, "ibd": ibd, **est}).groupby("ibd_bin", observed=False)
out = tab.mean().round(4).add_prefix("mean_").assign(pairs=tab.size())
out = out[out["pairs"] >= 21]
display(out)
fig, ax = plt.subplots(figsize=(6, 5))
for lab in est:
    ax.plot(out["mean_ibd"], out[f"mean_{lab}"], "o-", ms=4, label=lab)
ax.plot([0, 0.5], [0, 0.5], "k:", lw=0.8); ax.set(xscale="symlog", xlabel="iLASH IBD kinship (bin mean)", ylabel="kinship (bin mean)",
                                              title=f"{RUN_ID}: binned by IBD (bins with 21+ pairs)")
ax.legend(); plt.show()